# Självstudieövningar – Vecka 4

De här övningarna är till för att du ska repetera och befästa det vi har gått igenom under veckan samt introducera några nya begrepp på ett tryggt sätt.

Det innebär att vissa av uppgifterna kan du behöva söka efter information själv hur du löser. Här kan du använda Google eller AI om du vill men Python-dokumentationen är din bästa vän när du lär dig.

Länk: https://docs.python.org/3.13/

Läs först igenom vad som ska göras i respektive uppgift och fyll sedan i kodblocket nedanför, eller gör uppgiften i en separat fil.

**Detta lämnas inte in för rättning.** Arbeta i din egen takt, testa dig fram, använd dokumentationen när du kör fast och ta med eventuella frågor till nästa lektionstillfälle.

## (Kör först) Skapa veckans dataset

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # reproducible
n = 300
cities = np.array(["Göteborg","Malmö","Stockholm","Uppsala"])
genders = np.array(["F","M"])

age = rng.integers(18, 65, size=n)
gender = rng.choice(genders, size=n)
city = rng.choice(cities, size=n, p=[0.3,0.2,0.4,0.1])
steps = rng.normal(8000, 2000, size=n).clip(1000, 20000).astype(int)
sleep_hours = rng.normal(7.0, 1.2, size=n).clip(3.5, 11.0)
screen_time = rng.normal(4.0, 1.5, size=n).clip(0.5, 12.0)
calories = rng.normal(2200, 450, size=n).clip(1000, 4000)

# relationer: fler steg → lägre skärmtid, högre kalorier
screen_time = np.clip(screen_time - (steps - 8000)/8000 * 0.6, 0.5, 12.0)
calories    = np.clip(calories    + (steps - 8000)/8000 * 120, 1000, 4000)

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "city": city,
    "steps": steps,
    "sleep_hours": np.round(sleep_hours,2),
    "screen_time": np.round(screen_time,2),
    "calories": np.round(calories,0).astype(int)
})

# ca 2% saknade värden (NaN)
mask = rng.random(size=df.shape) < 0.02
for (r, c), m in np.ndenumerate(mask):
    if m:
        df.iat[r, c] = np.nan

df.to_csv("week4_data.csv", index=False)
print("Skapade week4_data.csv med", len(df), "rader.")

Skapade week4_data.csv med 300 rader.


### Om datasettet (week4_data.csv)
Det här är ett **syntetiskt** dataset från en fiktiv hälso-/livsstilsapp. Varje rad är en person/dag.

| Kolumn | Typ | Beskrivning | Ungefärligt intervall |
|---|---|---|---|
| `age` | int | Ålder | 18–64 |
| `gender` | kategori | Kön (F/M) | – |
| `city` | kategori | Stad | Göteborg, Malmö, Stockholm, Uppsala |
| `steps` | int | Antal steg/dygn | ~1 000–20 000 |
| `sleep_hours` | float | Sömn/dygn | ~3.5–11 |
| `screen_time` | float | Skärmtid/dygn | ~0.5–12 |
| `calories` | int | Energiförbrukning | ~1 000–4 000 |

**Noteringar:** Fast seed för reproducerbarhet, ~2% NaN, svag relation: fler `steps` → lägre `screen_time` men högre `calories` i snitt.

### Snabb kontroll
- Läs in `week4_data.csv` i `df`.
- Visa `df.head()` för att se de första raderna.
- Visa datatyper med `df.info()`.
- Räkna saknade värden per kolumn med en metod i Pandas.
- Skriv en kort kommentar (i en kodkommentar) om vilken kolumn som verkar ha flest NaN.

In [ ]:
df = pd.read_csv("week4_data.csv")
print(df.head())
print()
print(df.info())
print()
print(df.isna().sum()) # age seems to have most NaN values


## Block A – NumPy

### Uppgift 1 – Array-grunder
Skapa en följd av heltal och gör enkla operationer med **vektoriserad** aritmetik.
- Skapa en 1D-array med heltal 0..19.
- Skapa en ny array med kvadraterna av dessa tal (utan loop).
- Plocka ut de tal som är delbara med 3 genom ett boolean-filter.

In [ ]:
array1 = np.arange(20)
array2 = array1 **2
print(array2)
mask = array1 % 3 == 0
print(array1[mask])

### Uppgift 2 – Slicing & reshaping
Träna indexering i 2D.
- Skapa en array med tal 1..24.
- Forma om till en 4×6-matris.
- Plocka ut kolumnerna 2–4 (inklusive) och raderna 2–3 (inklusive) via slicing.

In [ ]:
array3 = np.arange(1, 25).reshape(4, 6)
print(array3)
print()
print(array3[1:3, 1:4])

### Uppgift 3 – Broadcasting
Använd broadcasting för att kombinera en matris och en vektor.
- Skapa en 3×3-matris med ettor.
- Skapa en vektor `[10, 20, 30]`.
- Lägg till vektorn **radvis** till matrisen och skriv ut resultatet.

In [ ]:
matrix1 = np.zeros((3, 3))
vector1 = [10, 20, 30]
print(matrix1 + vector1)


### Uppgift 4 – Felsökning (listor vs arrayer)
Koden nedan försöker summera elementvis men använder listor. Kör och rätta.
```python
import numpy as np
a = [1,2,3]
b = [4,5,6]
print(a + b)      # blir listkonkatenering
print(a * b)      # TypeError
```
**Mål:** gör om till arrayer så att addition/multiplikation blir elementvis.

In [ ]:
a = np.array([1,2,3])
b = np.array([4,5,6])

print(a + b)
print(a * b)


### Uppgift 5 – Simulera och jämför tärningar

Simulera två omgångar med 1 000 tärningskast och jämför resultaten.

- Skapa först en slumptalsgenerator med `np.random.default_rng(42)`.
- Använd generatorns metod `integers()` för att skapa två separata simuleringar.
- Varje simulering ska innehålla 1 000 utfall mellan 1 och 6.
- Räkna hur många ettor, tvåor och så vidare som förekommer i respektive simulering.
- Jämför simuleringarna genom att beräkna den absoluta skillnaden mellan frekvenserna.
- Skriv ut resultatet som en liten tabell.  

*(tips: en funktion i NumPy: `np.bincount` kan räkna förekomster per heltal)*

In [ ]:
rng = np.random.default_rng(42)

sim1 = rng.integers(1,7, size=1000)
sim2 = rng.integers(1,7, size=1000)

frequency1 = np.bincount(sim1)[1:]
frequency2 = np.bincount(sim2)[1:]

difference = np.abs(frequency1 - frequency2) # np.abs for absolute always positive

print(f"Tärningspoäng   Simulering 1   Simulering 2    Absolut skillnad")
print("-" * 63)
for i in range(6):
    print(f"{i+1}                  {frequency1[i]}              {frequency2[i]}               {difference[i]}")

## Block B – Pandas (inkl. rensa data)

### Uppgift 6 – Skapa och inspektera DataFrame
- Skapa en DataFrame **manuellt** från en dictionary (3–4 rader), t.ex. namn/ålder/stad.
- Visa de första raderna.
- Visa datatyper och en numerisk sammanfattning.

In [ ]:
data_dict = {
        "Namn": ["Anna", "Björn", "Cecilia", "Daniel"],
        "Ålder":[32, 51, 28, 37],
        "Stad": ["Stockholm", "Göteborg", "Malmö", "Uppsala"],
        "Inköp_kr": [1500, 2300, 850, 4100]
        }

df2 = pd.DataFrame(data_dict)
df2.head()
df2.info()
df2.describe()


### Uppgift 7 – Välja & filtrera
- Läs `week4_data.csv`.
- Skapa en vy som bara innehåller kolumnerna `age`, `steps`, `city`.
- Filtrera fram rader där `steps` är större än 10 000 **och** `city` är Göteborg.
- Visa de 5 första raderna av resultatet.

In [ ]:
df3 = pd.read_csv("week4_data.csv")
df3_view = df3[["city", "age", "steps"]]
mask = (df3_view["steps"] > 10000) & (df3_view["city"] == "Göteborg")

df3_view[mask].head()

### Uppgift 8 – Sortera & skapa kolumner
- Sortera data fallande på `steps` (störst först).
- Skapa en ny kolumn `steps_k` som visar steg i tusental (avrunda till 1 decimal).
- Visa de 5 första raderna efter sorteringen.

In [ ]:
df3 = df3.sort_values(by="steps", ascending=False)
df3 = df3.assign(steps_k = (df3["steps"] / 1000).round(1))

df3.head()

### Uppgift 9 – Saknade värden
- Räkna antalet saknade värden per kolumn.
- Fyll saknade i `sleep_hours` med **medianen**.
- Fyll saknade i `steps` med **medelvärdet**.
- Kontrollera igen att saknade värden minskat för dessa kolumner.

In [ ]:
print(df3.isna().sum())

median_value = df3["sleep_hours"].median()
mean_value = df3["steps"].mean()

df3["sleep_hours"] = df3["sleep_hours"].fillna(median_value)
df3["steps"] = df3["steps"].fillna(mean_value)
print(df3.isna().sum())


### Uppgift 10 – Dubbletter

Skapa först en version av datasetet som innehåller några dubbletter:

```python
df_with_duplicates = pd.concat(
    [df, df.iloc[:3]],
    ignore_index=True
)
```
Arbeta därefter med df_with_duplicates:

- räkna hur många rader som är dubbletter,
- ta bort dubblettraderna,
- visa radantalet före och efter borttagningen,
- kontrollera att det rensade datasetet har samma antal rader som det ursprungliga df

In [8]:
df_with_duplicates = pd.concat(
    [df, df.iloc[:3]],
    ignore_index=True
)

print("Dublettrader:", df_with_duplicates.duplicated().sum())
df_cleaned = df_with_duplicates.drop_duplicates().reset_index(drop=True)
print(f"Original length: {len(df)}")
print(f"Duplicated length: {len(df_with_duplicates)}")
print(f"Cleaned length: {len(df_cleaned)}")
      

Dublettrader: 3
Original length: 300
Duplicated length: 303
Cleaned length: 300


### Uppgift 11 – Typer och säker konvertering

Skapa först en kopia av datasetet där kolumnen `steps` lagras som text och ett av värdena är ogiltigt:

```python
df_dirty = df.copy()
df_dirty["steps"] = df_dirty["steps"].astype("string")
df_dirty.loc[0, "steps"] = "okänd"
```

Arbeta därefter med `df_dirty`:  
- kontrollera datatypen för `steps`,
- konvertera `steps` till numeriska värden på ett **säkert** sätt,
- se till att text som inte kan tolkas som ett tal blir `NaN`,
- fyll saknade värden i `steps` med kolumnens medelvärde,
- jämför datatyp och antal saknade värden före och efter konverteringen.

*(tips: titta på en Pandas-funktion som börjar `pd.to_num...`)*

### Uppgift 12 – Rengöringspipeline
Gör en liten pipeline som kopplar ihop allt:
- Hantera saknade värden (se uppgift 9/11).
- Skapa en ny kolumn `active_ratio = steps / (screen_time * 1000)`.
- Filtrera bort orimliga rader (t.ex. `sleep_hours < 4` eller `> 12`).
- Gruppera per `city` och skriv ut medelvärden för `steps` och `active_ratio`.

## Block C – NumPy + Pandas (deskriptiv statistik)

### Uppgift 13 – Centrala mått med NumPy
- Hämta kolumnen `steps` som en NumPy-array med `df["steps"].to_numpy()`.
- Ta bort eller ignorera saknade värden.
- Räkna ut medelvärde, median och stickprovsstandardavvikelse.
- Skriv ut resultaten med tydliga beskrivningar.

Använd `ddof=1` när du beräknar standardavvikelsen, eftersom observationerna här behandlas som ett stickprov.

*(tips: `np.mean`, `np.median`, `np.std`)*

### Uppgift 14 – Fem-tals-sammanfattning
För `sleep_hours`:
- Räkna fram min, Q1 (25%), median, Q3 (75%), max.
- Ignorera saknade värden först.  
*(tips: det finns en NumPy-funktion som heter `np.percentile`)*

### Uppgift 15 – IQR och möjliga extremvärden

Undersök kolumnen `sleep_hours`:

- ta bort eller ignorera saknade värden,
- beräkna Q1 och Q3,
- beräkna `IQR = Q3 - Q1`,
- beräkna gränserna `low = Q1 - 1.5 * IQR` och `high = Q3 + 1.5 * IQR`,
- skapa en DataFrame `outliers_df` med rader som ligger utanför intervallet,
- skriv ut hur många möjliga extremvärden som hittades,
- visa kolumnen `sleep_hours` för dessa rader.

Värden utanför IQR-gränserna är möjliga extremvärden och behöver inte automatiskt vara felaktiga.

### Uppgift 16 – Felsökning (typkonvertering)
Koden försöker räkna medelvärde på blandade typer. Kör och rätta.
```python
import numpy as np
vals = ["3", "5", "7", None]
print(np.mean(vals))  # TypeError
```
**Mål:** gör om värdena till tal och ignorera ogiltiga poster.

In [ ]:
# Din rättade kod här

### Uppgift 17 – Sammanfattning per stad

Skapa en gruppsammanställning med `city` som index och följande kolumner:

- `steps_mean` – medelvärdet av `steps`,
- `sleep_median` – medianen av `sleep_hours`,
- `screen_std` – standardavvikelsen för `screen_time`.

Sortera resultatet fallande efter `steps_mean`.

**Tips:** Pandas har så kallad *named aggregation*, där namn, ursprungskolumn och beräkning anges tillsammans:

```python
new_name=("original_column", "calculation")
```

## Block D – Matplotlib

### Uppgift 18 – Stapeldiagram från frekvenser
- Räkna frekvens av `city` med `value_counts()`.
- Rita ett **stapeldiagram**.
- Lägg till titel, axelrubriker och rutnät för att göra grafen lätt att läsa.
*(tips: du kan antingen använda `DataFrame.plot(...)` eller `matplotlib.pyplot`)*

### Uppgift 19 – Fördelning (histogram)
- Rita ett **histogram** av `steps` (välj ett rimligt antal bins, t.ex. 20).
- Lägg till titel/axlar och ett rutnät.
- Skriv en kort kommentar i kod (t.ex. om fördelningen ser skev ut).

### Uppgift 20 – Samband (scatter)
- Skapa en **scatterplot** med `steps` på x-axeln och `calories` på y-axeln.
- Lägg till titel, axelrubriker och rutnät.
- Skriv en kort kommentar om du tycker sambandet verkar positivt/negativt/svagt.

### Uppgift 21 – Felsökning (plot)
Koden försöker rita ett histogram på en icke-existerande kolumn. Kör och rätta.
```python
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("week4_data.csv")
df["step"].plot(kind="hist", bins=10)
plt.show()
```
**Mål:** använd korrekt kolumn och visa grafen.

In [ ]:
# Din rättade kod här

### Uppgift 22 – Mini-dashboard
Skapa tre figurer:
1. Histogram av `sleep_hours`,
2. Stapeldiagram med frekvenser för `gender`,
3. Scatter `steps` vs `screen_time`.

Lägg på titlar/axlar/rutnät och spara **en** figur till fil (valfritt filnamn).